# Member 1 — Phase 3: Data Cleaning & Preprocessing

**Project:** Hotel Booking Demand — Cancellation Risk  
**Owner:** IT24101668 (Member 1)

This notebook performs the group's proposed Phase 3 cleaning steps on the raw hotel booking dataset.

**Important:** Feature decisions that affect modelling should be confirmed by the group before becoming the final shared preprocessing standard.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/raw/hotel_bookings.csv")
df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
df.head()


## 1. Check exact duplicate rows

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)


In [ ]:
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)


## 2. Remove post-outcome leakage fields

`reservation_status` and `reservation_status_date` describe the final reservation outcome/status. They are excluded from the predictive feature set to reduce target leakage risk.


In [ ]:
leakage_cols = ["reservation_status", "reservation_status_date"]
df = df.drop(columns=leakage_cols)
print("Removed:", leakage_cols)
print("Current shape:", df.shape)


## 3. Remove high-missingness identifier fields

`company` has extremely high missingness and `agent` is an identifier-like field with substantial missingness. The current proposed common preprocessing removes both.

This is a group-level modelling decision and should be confirmed before finalising the shared dataset.


In [ ]:
identifier_cols = ["company", "agent"]
missing_before = {c: df[c].isna().sum() for c in identifier_cols}
print("Missing values before removal:", missing_before)

df = df.drop(columns=identifier_cols)
print("Removed:", identifier_cols)
print("Current shape:", df.shape)


## 4. Handle missing values

In [ ]:
print("Missing children:", df["children"].isna().sum())
print("Missing country:", df["country"].isna().sum())


In [ ]:
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")

print("Remaining children missing:", df["children"].isna().sum())
print("Remaining country missing:", df["country"].isna().sum())


## 5. Check invalid numerical values

`adr` should not be negative because it represents the average daily rate. We inspect negative values before deciding how to handle them.


In [ ]:
negative_adr = df[df["adr"] < 0]
print("Negative ADR records:", len(negative_adr))
negative_adr[["hotel", "adr", "adults", "children", "babies", "is_canceled"]]


In [ ]:
# Replace the invalid negative ADR with missing, then impute using the hotel-specific median.
negative_adr_mask = df["adr"] < 0
df.loc[negative_adr_mask, "adr"] = np.nan

df["adr"] = df.groupby("hotel")["adr"].transform(
    lambda s: s.fillna(s.median())
)

print("Remaining missing ADR:", df["adr"].isna().sum())
print("Remaining negative ADR:", (df["adr"] < 0).sum())


## 6. Check zero-guest records

A record with zero adults, zero children and zero babies is treated as an invalid booking record for this project.


In [ ]:
guest_total = df["adults"] + df["children"] + df["babies"]
zero_guest_count = (guest_total == 0).sum()
print("Zero-guest records:", zero_guest_count)


In [ ]:
df = df.loc[guest_total != 0].copy()
print("Shape after zero-guest removal:", df.shape)


## 7. Final quality checks

In [ ]:
print("Final shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Total duplicate rows:", df.duplicated().sum())
print("Negative ADR:", (df["adr"] < 0).sum())
print("Zero-guest records:",
      ((df["adults"] + df["children"] + df["babies"]) == 0).sum())

print("\nRemaining columns:")
print(df.columns.tolist())


## 8. Phase 3 handoff

The resulting dataset is the proposed common cleaned dataset for the modelling phase.

Before the group treats it as final, confirm:
- treatment of `company`
- treatment of `agent`
- zero-guest records
- invalid negative `adr`

After agreement, all model members should use the same cleaned dataset and should not independently change the preprocessing rules.
